#  Contextual Retrieval

1. Contextual Retrieval의 개념과 기존 RAG의 한계 이해
2. LLM을 활용한 청크별 컨텍스트 생성 방법 습득
3. BM25 + Contextual Embedding 하이브리드 검색 구현
4. Contextual Retrieval의 성능 향상 효과 측정

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from pprint import pprint
from typing import List, Tuple

`(3) langfuse handler 설정`

In [ ]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

---

## 문제 제기: 청킹 시 컨텍스트 손실

### 기존 RAG의 한계

일반적인 RAG 시스템에서는 문서를 작은 청크로 분할하여 검색합니다. 하지만 이 과정에서 **중요한 맥락 정보가 손실**될 수 있습니다.

```mermaid
graph TD
    A["원본 문서<br/>테슬라 2023 Q3 실적 보고서"] -->|청킹| B["청크 50<br/>이 분기의 매출은 전년 동기 대비 8% 증가..."]
    B -->|❌ 기존 RAG| C["맥락 손실<br/>어떤 회사? 언제?"]
    A --> D[문서 전체 맥락]
    B --> E[컨텍스트 추가]
    D --> E
    E -->|✅ Contextual Retrieval| F["컨텍스트 + 청크<br/>테슬라 2023 Q3 매출 성과..."]
```

### 예시

원본 문서:
```
테슬라의 2023년 3분기 실적 보고서
...
[청크 50] 이 분기의 매출은 전년 동기 대비 8% 증가한 233억 달러를 기록했습니다.
```

**문제점**: 청크 50만 보면 "어떤 회사", "언제" 등의 맥락을 알 수 없습니다.

### Anthropic의 해결책: Contextual Retrieval

**핵심 아이디어**: 각 청크에 **문서 전체 맥락을 설명하는 컨텍스트를 추가**합니다.

```
[컨텍스트] 이 청크는 테슬라의 2023년 3분기 실적 보고서에서 매출 성과를 다룹니다.
[원본 청크] 이 분기의 매출은 전년 동기 대비 8% 증가한 233억 달러를 기록했습니다.
```

### 성능 향상 효과 (Anthropic 연구)

| 기법 | 검색 실패율 감소 |
|------|----------------|
| Contextual Embedding | 35% |
| + Contextual BM25 | 49% |
| + Reranker | 67% |


---

## 1. 데이터 준비

### 1-1) 샘플 문서 로드

In [ ]:
from langchain_core.documents import Document

# 긴 샘플 문서 (실제 사용 시에는 파일에서 로드)
sample_document = """
# 테슬라 2023년 연간 보고서

## 1. 회사 개요

테슬라(Tesla, Inc.)는 2003년 설립된 미국의 전기자동차 및 청정에너지 회사입니다. 
본사는 텍사스 오스틴에 위치해 있으며, 일론 머스크가 CEO로 재직 중입니다.
테슬라는 전기차, 에너지 저장 시스템, 태양광 패널 등을 제조 및 판매합니다.

## 2. 2023년 실적 요약

2023년 테슬라의 총 매출은 967억 달러를 기록했습니다.
전년 대비 19% 증가한 수치입니다.
자동차 부문 매출이 824억 달러로 전체의 85%를 차지했습니다.
에너지 저장 부문은 60억 달러의 매출을 달성했습니다.

## 3. 생산 및 판매

2023년 테슬라는 총 184만 5,985대의 차량을 생산했습니다.
이 중 Model Y가 120만 대 이상으로 가장 많이 생산되었습니다.
Model 3는 약 50만 대가 생산되어 두 번째로 많았습니다.
전 세계 인도량은 180만 8,581대를 기록했습니다.

## 4. 기술 개발

### 4.1 자율주행
FSD(Full Self-Driving) 베타 버전이 북미 전역으로 확대되었습니다.
누적 FSD 주행 거리가 10억 마일을 돌파했습니다.
Dojo 슈퍼컴퓨터를 활용한 AI 학습이 진행 중입니다.

### 4.2 배터리 기술
4680 배터리 셀의 대량 생산이 시작되었습니다.
텍사스 기가팩토리에서 주당 1,000만 셀 이상을 생산하고 있습니다.
배터리 비용을 kWh당 100달러 미만으로 낮추는 것이 목표입니다.

## 5. 미래 전망

2024년에는 차세대 저가 모델 출시가 예정되어 있습니다.
사이버트럭의 본격적인 양산도 시작될 예정입니다.
테슬라 로보택시 서비스 출시를 위한 준비가 진행 중입니다.
"""

# Document 객체로 변환
doc = Document(
    page_content=sample_document,
    metadata={"source": "tesla_annual_report_2023.md", "year": 2023}
)

print(f"문서 길이: {len(doc.page_content)} 글자")

### 1-2) 문서 청킹

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # 작은 청크로 분할
    chunk_overlap=50,    # 50자 중복
    separators=["\n\n", "\n", ". ",],
)

# 문서 분할
chunks = text_splitter.split_documents([doc])

print(f"생성된 청크 수: {len(chunks)}")
print("="*80)

for i, chunk in enumerate(chunks[:5]):
    print(f"\n[청크 {i+1}] ({len(chunk.page_content)}자)")
    print("-"*40)
    print(chunk.page_content[:200] + "..." if len(chunk.page_content) > 200 else chunk.page_content)

---

## 2. 컨텍스트 생성

### Contextual Retrieval의 핵심

LLM을 사용하여 각 청크에 대한 **간결한 맥락 설명**을 생성합니다.

### 2-1) 컨텍스트 생성 프롬프트

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LLM 초기화 (컨텍스트 생성용 - 가벼운 모델 사용)
context_llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

# Anthropic의 컨텍스트 생성 프롬프트 (한국어 버전)
context_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 문서의 청크에 맥락을 추가하는 전문가입니다.
주어진 청크가 전체 문서에서 어떤 위치에 있고 무엇에 대한 내용인지 간결하게 설명하세요.
설명은 50-100자 이내로 작성하세요.
오직 맥락 설명만 출력하세요."""),
    ("user", """<document>
{whole_document}
</document>

위 문서에서 아래 청크의 맥락을 설명해주세요:

<chunk>
{chunk_content}
</chunk>

맥락 설명:""")
])

# 컨텍스트 생성 체인
context_chain = context_prompt | context_llm | StrOutputParser()

print("컨텍스트 생성 체인 준비 완료")

### 2-2) 배치 컨텍스트 생성

In [ ]:
# 각 청크에 대해 컨텍스트 생성
def generate_contexts(chunks: List[Document], whole_document: str) -> List[str]:
    """청크들에 대한 컨텍스트를 배치로 생성합니다."""
    inputs = [
        {"whole_document": whole_document, "chunk_content": chunk.page_content}
        for chunk in chunks
    ]
    
    # 배치 처리로 효율적으로 생성
    contexts = context_chain.batch(
        inputs, 
        config={"max_concurrency": 5, "callbacks": [langfuse_handler]}
    )
    
    return contexts

# 컨텍스트 생성 실행
print("컨텍스트 생성 중...")
contexts = generate_contexts(chunks, sample_document)

print(f"\n생성된 컨텍스트 수: {len(contexts)}")
print("="*80)

for i, (chunk, context) in enumerate(zip(chunks[:5], contexts[:5])):
    print(f"\n[청크 {i+1}]")
    print(f"원본: {chunk.page_content[:100]}...")
    print(f"컨텍스트: {context}")

### 2-3) Contextual 청크 생성

In [ ]:
# 컨텍스트가 추가된 청크 생성
contextual_chunks = []

for i, (chunk, context) in enumerate(zip(chunks, contexts)):
    # 컨텍스트 + 원본 청크 결합
    contextual_content = f"[맥락] {context}\n\n{chunk.page_content}"
    
    contextual_chunk = Document(
        page_content=contextual_content,
        metadata={
            **chunk.metadata,
            "chunk_id": i,
            "original_content": chunk.page_content,
            "context": context,
        }
    )
    contextual_chunks.append(contextual_chunk)

print(f"Contextual 청크 생성 완료: {len(contextual_chunks)}개")
print("="*80)

# 예시 출력
print("\n[Contextual 청크 예시]")
print(contextual_chunks[3].page_content)

---

## 3. Contextual Embedding 검색

컨텍스트가 추가된 청크를 임베딩하여 검색합니다.

### 3-1) 벡터 저장소 생성

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 일반 청크 벡터 저장소 (비교용)
normal_vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="normal_chunks",
    persist_directory="./chroma_db"
)

# Contextual 청크 벡터 저장소
contextual_vectorstore = Chroma.from_documents(
    documents=contextual_chunks,
    embedding=embeddings,
    collection_name="contextual_chunks",
    persist_directory="./chroma_db"
)

print("벡터 저장소 생성 완료")
print(f"- 일반 청크: {normal_vectorstore._collection.count()}개")
print(f"- Contextual 청크: {contextual_vectorstore._collection.count()}개")

### 3-2) 검색 비교 테스트

In [ ]:
# 테스트 쿼리
test_queries = [
    "테슬라의 2023년 총 매출은 얼마인가?",
    "Model Y 생산량은 몇 대인가?",
    "4680 배터리 셀 생산량은?",
]

# Retriever 생성
normal_retriever = normal_vectorstore.as_retriever(search_kwargs={"k": 3})
contextual_retriever = contextual_vectorstore.as_retriever(search_kwargs={"k": 3})

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # 일반 검색
    normal_results = normal_retriever.invoke(query)
    print(f"\n[일반 검색 결과]")
    for i, doc in enumerate(normal_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")
    
    # Contextual 검색
    contextual_results = contextual_retriever.invoke(query)
    print(f"\n[Contextual 검색 결과]")
    for i, doc in enumerate(contextual_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")

---

## 4. Contextual BM25 검색

BM25는 정확한 키워드 매칭에 강점이 있습니다. 특히 **고유명사, 숫자, 에러 코드** 등의 검색에 효과적입니다.

### 4-1) BM25 Retriever 설정

In [ ]:
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi

# Kiwi 한국어 형태소 분석기 초기화
kiwi = Kiwi()

# 사용자 정의 단어 추가 (고유명사)
kiwi.add_user_word('테슬라', 'NNP')

# 한국어 토크나이저 전처리 함수
def kiwi_preprocess_func(text):
    """Kiwi 형태소 분석기를 사용한 토큰화"""
    return [t.form for t in kiwi.tokenize(text)]

# 일반 BM25 Retriever (Kiwi 토크나이저 적용)
normal_bm25 = BM25Retriever.from_documents(
    documents=chunks,
    preprocess_func=kiwi_preprocess_func,
    k=3
)

# Contextual BM25 Retriever (Kiwi 토크나이저 적용)
contextual_bm25 = BM25Retriever.from_documents(
    documents=contextual_chunks,
    preprocess_func=kiwi_preprocess_func,
    k=3
)

print("BM25 Retriever 생성 완료 (Kiwi 토크나이저 적용)")

### 4-2) BM25 검색 비교

In [ ]:
# 정확한 키워드가 필요한 쿼리
keyword_queries = [
    "967억 달러",
    "FSD 베타",
    "Dojo 슈퍼컴퓨터",
]

for query in keyword_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # 일반 BM25
    normal_results = normal_bm25.invoke(query)
    print(f"\n[일반 BM25]")
    for i, doc in enumerate(normal_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")
    
    # Contextual BM25
    contextual_results = contextual_bm25.invoke(query)
    print(f"\n[Contextual BM25]")
    for i, doc in enumerate(contextual_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")

---

## 5. 하이브리드 검색 (Embedding + BM25)

**의미 기반 검색(Embedding)** 과 **키워드 기반 검색(BM25)** 을 결합하여 최상의 결과를 얻습니다.

### 5-1) EnsembleRetriever 설정

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

# 일반 하이브리드 Retriever
normal_hybrid = EnsembleRetriever(
    retrievers=[normal_retriever, normal_bm25],
    weights=[0.5, 0.5],  # Embedding과 BM25 동일 가중치
)

# Contextual 하이브리드 Retriever
contextual_hybrid = EnsembleRetriever(
    retrievers=[contextual_retriever, contextual_bm25],
    weights=[0.5, 0.5],
)

print("하이브리드 Retriever 생성 완료")

### 5-2) 하이브리드 검색 테스트

In [ ]:
# 다양한 유형의 쿼리
hybrid_queries = [
    "테슬라의 연간 매출 실적은?",           # 의미 기반
    "4680 배터리 kWh당 목표 가격",          # 키워드 기반
    "2024년 출시 예정인 새로운 모델은?",     # 혼합
]

for query in hybrid_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # 일반 하이브리드
    normal_results = normal_hybrid.invoke(query)
    print(f"\n[일반 하이브리드] 결과 {len(normal_results)}개")
    for i, doc in enumerate(normal_results[:2]):
        print(f"  {i+1}. {doc.page_content[:80]}...")
    
    # Contextual 하이브리드
    contextual_results = contextual_hybrid.invoke(query)
    print(f"\n[Contextual 하이브리드] 결과 {len(contextual_results)}개")
    for i, doc in enumerate(contextual_results[:2]):
        # 원본 내용 표시
        original = doc.metadata.get('original_content', doc.page_content[:80])
        print(f"  {i+1}. {original[:80]}...")

---

## 6. Reranker 추가

하이브리드 검색 결과를 **Reranker로 재순위화**하여 최종 성능을 극대화합니다.

### 6-1) Reranker 설정

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Cross-Encoder 모델 초기화
cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)

# Contextual Hybrid + Reranker
contextual_rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=contextual_hybrid,
)

print("Reranker 설정 완료")

### 6-2) 최종 검색 테스트

In [ ]:
# 최종 검색 테스트
final_queries = [
    "테슬라의 자율주행 기술 현황은?",
    "2023년 차량 생산량과 인도량은?",
    "배터리 비용 절감 목표는?",
]

for query in final_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # Contextual Hybrid + Reranker
    results = contextual_rerank_retriever.invoke(query, config={"callbacks": [langfuse_handler]})
    
    print(f"\n[Contextual Hybrid + Reranker] Top {len(results)}개")
    for i, doc in enumerate(results):
        original = doc.metadata.get('original_content', doc.page_content)
        context = doc.metadata.get('context', 'N/A')
        print(f"\n  [{i+1}] 맥락: {context}")
        print(f"      내용: {original[:100]}...")

---

## 7. RAG 체인 구성

Contextual Retrieval을 활용한 완전한 RAG 시스템을 구축합니다.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# 답변 생성용 LLM
answer_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# RAG 프롬프트
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 문서 기반 질의응답 전문가입니다.
주어진 문맥을 바탕으로 질문에 정확하게 답변하세요.
문맥에 없는 내용은 "문서에서 해당 정보를 찾을 수 없습니다."라고 답변하세요."""),
    ("user", """문맥:
{context}

질문: {question}

답변:""")
])

# 문서 포맷팅 함수
def format_docs(docs):
    formatted = []
    for doc in docs:
        # 원본 내용과 맥락 모두 포함
        original = doc.metadata.get('original_content', doc.page_content)
        context = doc.metadata.get('context', '')
        if context:
            formatted.append(f"[맥락: {context}]\n{original}")
        else:
            formatted.append(original)
    return "\n\n---\n\n".join(formatted)

# RAG 체인 구성
rag_chain = (
    {"context": contextual_rerank_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | answer_llm
    | StrOutputParser()
)

print("RAG 체인 구성 완료")

In [ ]:
# RAG 체인 실행
questions = [
    "테슬라의 2023년 매출 구성을 설명해주세요.",
    "FSD의 현재 상황은 어떤가요?",
    "2024년에 예정된 신제품은 무엇인가요?",
    "4680 배터리의 생산 현황과 목표는?",
]

for question in questions:
    print(f"\n{'='*80}")
    print(f"Q: {question}")
    print("-"*80)
    
    answer = rag_chain.invoke(question, config={"callbacks": [langfuse_handler]})
    print(f"A: {answer}")

---

## 8. 성능 비교 요약

In [ ]:
# 다양한 Retriever 성능 비교 (Kiwi 기반)
from kiwipiepy import Kiwi

retrievers_to_compare = {
    "일반 Embedding": normal_retriever,
    "일반 BM25": normal_bm25,
    "일반 Hybrid": normal_hybrid,
    "Contextual Embedding": contextual_retriever,
    "Contextual BM25": contextual_bm25,
    "Contextual Hybrid": contextual_hybrid,
    "Contextual + Reranker": contextual_rerank_retriever,
}

# 테스트 쿼리와 기대 키워드
# 핵심: 컨텍스트에만 있고 원본 청크에는 없는 키워드로 검색
# k=1로 설정하여 정확한 청크 매칭 테스트
test_queries = [
    # 직접적인 쿼리 (일반 검색도 가능) - 원본 청크에 키워드 존재
    {"query": "테슬라의 2023년 자동차 부문 매출은?", "expected": "824억 달러"},
    {"query": "4680 배터리 생산량은?", "expected": "1,000만 셀"},
    
    # 맥락 의존적 쿼리 - 컨텍스트에만 있는 표현 사용
    # 원본 청크: "FSD 베타 버전이... Dojo 슈퍼컴퓨터를 활용한 AI 학습"
    # 컨텍스트: "기술 개발 부문, 자율주행과 AI 학습 관련 최신 성과"
    {"query": "테슬라의 최신 성과와 진행 상황", "expected": "Dojo"},
    
    # 원본 청크: "4680 배터리 셀의 대량 생산... kWh당 100달러"
    # 컨텍스트: "기술 혁신과 미래 전략의 핵심 내용"
    {"query": "테슬라의 기술 혁신 핵심 내용", "expected": "100달러"},
    
    # 원본 청크: "2023년 테슬라의 총 매출은 967억 달러... 에너지 저장 부문"
    # 컨텍스트: "재무 성과와 매출, 생산량에 대한 핵심 내용을 요약"
    {"query": "테슬라 재무 성과 요약", "expected": "967억 달러"},
    
    # 원본 청크: "Tesla, Inc.는 2003년 설립... 일론 머스크가 CEO"
    # 컨텍스트: "회사의 설립 배경, 주요 사업 분야"
    {"query": "테슬라 설립 배경과 사업 분야", "expected": "2003년"},
]

kiwi = Kiwi()

def evaluate_contextual_retrievers(retrievers: dict, test_queries: list, k: int = 1):
    """Contextual vs Non-Contextual Retriever 성능 비교
    
    k=1로 설정하여 가장 관련성 높은 1개 청크만 검색.
    문서가 작을 때 (6개 청크) k=3이면 50%를 검색하므로 변별력이 낮음.
    """
    results_dict = {}
    
    for name, retriever in retrievers.items():
        correct = 0
        total = len(test_queries)
        
        for test in test_queries:
            query = test["query"]
            expected = test["expected"]
            
            try:
                results = retriever.invoke(query)[:k]
                all_content = " ".join([
                    doc.metadata.get('original_content', '') or doc.page_content 
                    for doc in results
                ])
                
                # 기대 키워드가 검색 결과에 포함되어 있는지 확인
                if expected in all_content:
                    correct += 1
            except Exception as e:
                pass
        
        accuracy = correct / total
        results_dict[name] = {"accuracy": accuracy, "correct": correct, "total": total}
    
    return results_dict

print("="*80)
print("Contextual Retrieval 성능 비교 (k=1, Top-1 정확도)")
print("="*80)

eval_results = evaluate_contextual_retrievers(retrievers_to_compare, test_queries, k=1)

for name, result in eval_results.items():
    status = "✅" if result["accuracy"] >= 0.5 else "❌"
    print(f"{status} {name:25s}: {result['accuracy']:.1%} ({result['correct']}/{result['total']})")

# 일반 vs Contextual 비교
print("\n" + "="*80)
print("일반 vs Contextual 비교 요약")
print("="*80)

normal_avg = sum([eval_results[k]["accuracy"] for k in ["일반 Embedding", "일반 BM25", "일반 Hybrid"]]) / 3
contextual_avg = sum([eval_results[k]["accuracy"] for k in ["Contextual Embedding", "Contextual BM25", "Contextual Hybrid"]]) / 3
best_result = eval_results["Contextual + Reranker"]["accuracy"]

print(f"일반 검색 평균 정확도:      {normal_avg:.1%}")
print(f"Contextual 검색 평균 정확도: {contextual_avg:.1%}")
print(f"Contextual + Reranker:      {best_result:.1%}")

if normal_avg > 0:
    improvement = ((contextual_avg - normal_avg) / normal_avg * 100)
    print(f"\n→ Contextual 방식이 일반 방식 대비 {improvement:.1f}% 향상")
else:
    print(f"\n→ 일반 검색 정확도가 0%이므로 비율 계산 불가")

---

## 연습문제

다음 연습문제를 통해 Contextual Retrieval에 대한 이해를 확인해 보세요.

### 문제 1: 문맥 정보 생성 프롬프트

아래 코드의 빈칸을 채워 청크에 문맥 정보를 추가하는 프롬프트를 완성하세요.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 문맥 정보 생성 프롬프트
context_prompt = ChatPromptTemplate.from_template(
    """다음은 전체 문서입니다:
<document>
{____}
</document>

다음은 문서의 일부 청크입니다:
<chunk>
{____}
</chunk>

이 청크가 전체 문서에서 어떤 맥락에 위치하는지 간단히 설명해주세요.
검색에 도움이 되도록 핵심 키워드와 주제를 포함해주세요.
응답은 2-3문장으로 작성하세요."""
)

# 힌트: 첫 번째 빈칸은 전체 문서, 두 번째 빈칸은 청크를 위한 변수명

### 문제 2: Contextual 청크 생성

아래 코드의 빈칸을 채워 문맥 정보가 포함된 청크를 생성하세요.

In [ ]:
from langchain_core.documents import Document

# 원본 청크와 생성된 문맥 정보 (예시)
original_chunk = "Tesla는 2024년에 새로운 배터리 기술을 발표했습니다."
generated_context = "이 청크는 Tesla의 배터리 기술 발전에 관한 섹션에서 발췌되었습니다."

# Contextual 청크 생성
# 문맥 정보를 청크 앞에 추가하는 형식
contextual_content = f"[맥락] {____}\n\n{____}"  # 힌트: generated_context, original_chunk

# Document 객체 생성
contextual_doc = ____(
    page_content=contextual_content,
    metadata={"source": "Tesla_KR.md", "has_context": True}
)

print(f"Contextual 청크:\n{contextual_doc.page_content}")

--- 
# **[실습]**

### 실습 목표

Contextual Retrieval을 실제 문서에 적용하여 검색 성능을 개선합니다.

### 난이도별 가이드

**기본 난이도:**
- 제공된 샘플 문서에 Contextual Retrieval 적용
- 일반 검색 vs Contextual 검색 성능 비교
- 3개 이상의 쿼리로 테스트

**중급 난이도:**
- 자체 문서(예: 기술 문서, 위키피디아) 활용
- Hybrid 검색 (Embedding + BM25) 구현
- 가중치 조합 실험 (0.3:0.7, 0.5:0.5, 0.7:0.3)

**고급 난이도:**
- 다국어 문서에 Contextual Retrieval 적용
- Reranker와 결합하여 최적 파이프라인 구성
- 검색 성능 지표 (HitRate, MRR) 측정 및 비교

`(1) 문서 준비 및 청킹`

자신의 문서를 로드하고 청크로 분할합니다.

In [ ]:
# 여기에 코드를 작성하세요.


`(2) 컨텍스트 생성`

각 청크에 대해 맥락 설명을 생성합니다.

In [ ]:
# 여기에 코드를 작성하세요.


`(3) 하이브리드 검색 및 평가`

Embedding + BM25 + Reranker 파이프라인을 구성하고 성능을 평가합니다.

In [ ]:
# 여기에 코드를 작성하세요.
